<a href="https://colab.research.google.com/github/hsuancheyang/115-1-AI_Fundamentals/blob/main/%E5%B0%8D%E8%A9%B1%E6%A9%9F%E5%99%A8%E4%BA%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 設定 Google AI Studio API 金鑰

你需要一個 API 金鑰才能使用 Gemini API。如果您還沒有，請在 [Google AI Studio](https://aistudio.google.com/app/apikey) 中建立一個金鑰。

在 Colab 中，您可以點擊左側面板中的 "🔑" 圖標，將金鑰儲存到秘密管理器中。請將其命名為 `GOOGLE_API_KEY`。然後，您可以像下面程式那樣將金鑰傳遞給 SDK：

In [1]:
# 導入 Python SDK
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 2. 初始化 Gemini 模型

在進行任何 API 呼叫之前，您需要初始化生成模型。我們將使用 `gemini-2.5-flash` 模型。

In [2]:
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

## 3. 建立網頁版對話機器人介面 (使用 Gradio)

我們使用 `gradio` 函式庫來建立一個簡單的網頁介面，可以輸入系統提示`system_prompt`（人設）、調整 `top_p` (p 值) 和 `temperature` (回應溫度)，並進行多輪對話。

首先，安裝 `gradio`：

In [4]:
!pip install -q gradio

接下來是聊天機器人的程式碼。它包含一個 `chat_session` 函數，用於處理對話邏輯，以及一個 `gradio` 介面來展示它。

In [8]:
import gradio as gr
import google.generativeai as genai

def chat_session(user_message, history, system_prompt, top_p, temperature):
    global gemini_model

    generation_config = {
        "temperature": temperature,
        "top_p": top_p,
    }

    model_history_for_chat = []
    if system_prompt:
        model_history_for_chat.append({'role': 'user', 'parts': [system_prompt]})
        model_history_for_chat.append({'role': 'model', 'parts': ['好的，我已收到您的指示。']})

    for human_message, bot_message in history:
        model_history_for_chat.append({'role': 'user', 'parts': [human_message]})
        model_history_for_chat.append({'role': 'model', 'parts': [bot_message]})

    chat = gemini_model.start_chat(history=model_history_for_chat)

    try:
        response = chat.send_message(user_message, generation_config=generation_config)
        return response.text
    except Exception as e:
        print(f"Error sending message: {e}")
        return f"發生錯誤: {e}"

# Define initial values for reset
INITIAL_SYSTEM_PROMPT = ""
INITIAL_TOP_P = 0.9
INITIAL_TEMPERATURE = 0.7

with gr.Blocks(theme="soft", title="AI 助理聊天機器人") as demo:
    gr.Markdown("# My 聊天機器人")
    gr.Markdown("您可以設定系統提示(人格設定)、調整選字策略 (top_p) 和回應溫度 (temperature)。")

    with gr.Row():
        with gr.Column(scale=1):
            system_prompt_input = gr.Textbox(
                label="系統提示 (System Prompt)",
                placeholder="設定聊天機器人的人設，例如：你是一個樂觀的AI助理。",
                lines=3,
                value=INITIAL_SYSTEM_PROMPT
            )
            top_p_slider = gr.Slider(
                minimum=0.0, maximum=1.0, step=0.01,
                value=INITIAL_TOP_P, label="選字策略 (Top P)",
                info="調整詞彙選擇的多樣性。值越大，選擇越多樣"
            )
            temperature_slider = gr.Slider(
                minimum=0.0, maximum=1.0, step=0.01,
                value=INITIAL_TEMPERATURE, label="回應溫度 (Temperature)",
                info="調整回應的隨機性。值越低會產生更可預測的回應，值越高越發散熱情。"
            )
            clear_all_btn = gr.Button("清除所有 (Clear All)")

        with gr.Column(scale=2):
            chatbot_component = gr.Chatbot(height=300)
            msg = gr.Textbox(placeholder="輸入您的訊息...", container=False, scale=7)
            send_btn = gr.Button("送出")

    # Define the response function for messages
    def respond(message, chat_history, system_prompt, top_p, temperature):
        # Call the original chat_session function
        bot_message = chat_session(message, chat_history, system_prompt, top_p, temperature)
        chat_history.append((message, bot_message))
        return chat_history

    # Clear function
    def clear_all_states_func():
        return (
            [], # clear chatbot history
            INITIAL_SYSTEM_PROMPT, # reset system prompt
            INITIAL_TOP_P, # reset top_p
            INITIAL_TEMPERATURE, # reset temperature
            ""   # clear message textbox
        )

    clear_all_btn.click(
        clear_all_states_func,
        outputs=[chatbot_component, system_prompt_input, top_p_slider, temperature_slider, msg]
    )

    # Setup interaction for message submission
    msg.submit(
        respond,
        [msg, chatbot_component, system_prompt_input, top_p_slider, temperature_slider],
        chatbot_component,
    ).then(
        lambda: gr.update(value=""),
        inputs=None,
        outputs=msg,
    )

    send_btn.click(
        respond,
        [msg, chatbot_component, system_prompt_input, top_p_slider, temperature_slider],
        chatbot_component,
    ).then(
        lambda: gr.update(value=""),
        inputs=None,
        outputs=msg,
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_6957/2815266812.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme="soft", title="AI 助理聊天機器人") as demo:
/tmp/ipykernel_6957/2815266812.py:60: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_component = gr.Chatbot(height=300)
/tmp/ipykernel_6957/2815266812.py:60: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_component = gr.Chatbot(height=300)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://790e6d3b5f0d6b7478.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1111.48ms


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://790e6d3b5f0d6b7478.gradio.live
